In [15]:
# HW02 - 优化版
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

# 设备配置 - 优先使用GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

使用设备: cpu


In [16]:
# ========================================
# 2.2 单隐藏层 MLP
# ========================================
print("\n=== 2.2 单隐藏层 MLP ===")

# 数据加载（优化：num_workers加速）
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_set = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.FashionMNIST('./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)

# 参数初始化
input_dim = 784
hidden_dim = 256
output_dim = 10

# 使用PyTorch参数封装（更高效）
W1 = torch.randn(input_dim, hidden_dim, device=device) * 0.01
b1 = torch.zeros(hidden_dim, device=device)
W2 = torch.randn(hidden_dim, output_dim, device=device) * 0.01
b2 = torch.zeros(output_dim, device=device)

params = [W1, b1, W2, b2]
for p in params:
    p.requires_grad_(True)

# 前向传播
def forward(x):
    x = x.view(x.shape[0], -1).to(device)
    h = torch.relu(x @ W1 + b1)
    out = h @ W2 + b2
    return out

# 训练（优化：使用内置交叉熵）
def train(epochs=3, lr=0.1):
    for epoch in range(epochs):
        total_loss = 0.0
        for x, y in train_loader:
            y = y.to(device)
            y_pred = forward(x)
            
            # 使用PyTorch内置交叉熵（更高效）
            loss = torch.nn.functional.cross_entropy(y_pred, y)
            total_loss += loss.item()
            
            loss.backward()
            
            with torch.no_grad():
                for p in params:
                    p -= lr * p.grad
                    p.grad.zero_()
        
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

train(epochs=3)

# 测试
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            y = y.to(device)
            y_pred = forward(x)
            _, predicted = torch.max(y_pred, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

test()


=== 2.2 单隐藏层 MLP ===


100%|██████████| 26.4M/26.4M [00:09<00:00, 2.84MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 139kB/s]
100%|██████████| 4.42M/4.42M [00:03<00:00, 1.39MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


Epoch 1, Loss: 0.6483
Epoch 2, Loss: 0.4391
Epoch 3, Loss: 0.3945
Test Accuracy: 85.44%


In [17]:
# ========================================
# 3.2 L2正则化和Dropout
# ========================================
print("\n=== 3.2 L2正则化和Dropout ===")

# Dropout函数
def dropout_layer(X, dropout):
    mask = (torch.rand_like(X) > dropout).float()
    return mask * X / (1.0 - dropout)

# 带Dropout的前向传播
def forward_dropout(x, is_training=True, dropout=0.5):
    x = x.view(x.shape[0], -1).to(device)
    h = torch.relu(x @ W1 + b1)
    if is_training:
        h = dropout_layer(h, dropout)
    out = h @ W2 + b2
    return out

# 带正则化和Dropout的训练
def train_with_reg(epochs=3, lr=0.1, weight_decay=0.001, dropout=0.5):
    for epoch in range(epochs):
        total_loss = 0.0
        for x, y in train_loader:
            y = y.to(device)
            y_pred = forward_dropout(x, is_training=True, dropout=dropout)
            
            # 交叉熵 + L2正则化
            loss = torch.nn.functional.cross_entropy(y_pred, y)
            reg = weight_decay * (torch.sum(W1**2) + torch.sum(W2**2))
            total_loss += (loss + reg).item()
            
            (loss + reg).backward()
            
            with torch.no_grad():
                for p in params:
                    p -= lr * p.grad
                    p.grad.zero_()
        
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

print("有权重衰减和Dropout:")
train_with_reg(epochs=3)


=== 3.2 L2正则化和Dropout ===
有权重衰减和Dropout:
Epoch 1, Loss: 0.5095
Epoch 2, Loss: 0.4877
Epoch 3, Loss: 0.4789


In [18]:
# ========================================
# 4.2 数值稳定性
# ========================================
print("\n=== 4.2 数值稳定性 ===")

# 快速测试梯度
def test_grad_stability(activation='sigmoid', init_std=1.0):
    net = []
    for _ in range(10):  # 减少层数加速
        layer = torch.nn.Linear(256, 256).to(device)
        torch.nn.init.normal_(layer.weight, std=init_std)
        net.append(layer)
        net.append(torch.nn.Sigmoid() if activation == 'sigmoid' else torch.nn.ReLU())
    
    net = torch.nn.Sequential(*net)
    
    x = torch.randn(32, 256, device=device, requires_grad=True)
    y = net(x)
    y.sum().backward()
    
    norms = [m.weight.grad.norm().item() for i, m in enumerate(net) if isinstance(m, torch.nn.Linear)]
    print(f"前3层梯度范数: {norms[:3]}")
    print(f"后3层梯度范数: {norms[-3:]}")

print("Sigmoid + std=1.0 (梯度消失):")
test_grad_stability('sigmoid', 1.0)

print("\nReLU + Xavier (稳定):")
net = torch.nn.Sequential()
for _ in range(10):
    layer = torch.nn.Linear(256, 256).to(device)
    torch.nn.init.xavier_uniform_(layer.weight)
    net.append(layer)
    net.append(torch.nn.ReLU())

x = torch.randn(32, 256, device=device, requires_grad=True)
y = net(x)
y.sum().backward()
norms = [m.weight.grad.norm().item() for i, m in enumerate(net) if isinstance(m, torch.nn.Linear)]
print(f"梯度范数范围: [{min(norms):.2e}, {max(norms):.2e}]")



=== 4.2 数值稳定性 ===
Sigmoid + std=1.0 (梯度消失):
前3层梯度范数: [826.3566284179688, 534.724365234375, 418.69586181640625]
后3层梯度范数: [222.3847198486328, 264.7270812988281, 280.18292236328125]

ReLU + Xavier (稳定):
梯度范数范围: [7.68e+01, 2.67e+02]


In [19]:
# ========================================
# 5.2 协变量偏移校正
# ========================================
print("\n=== 5.2 协变量偏移校正 ===")

# 使用NumPy快速实现
np.random.seed(42)
X_train = np.random.normal(-1, 1, 1000)
y_train = 2 * X_train + np.random.normal(0, 0.1, 1000)
X_test = np.random.normal(2, 1, 500)
y_test = 2 * X_test + np.random.normal(0, 0.1, 500)

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error

# 基线模型
model = LinearRegression()
model.fit(X_train.reshape(-1, 1), y_train)
mse_before = mean_squared_error(y_test, model.predict(X_test.reshape(-1, 1)))

# 域分类器
clf = LogisticRegression()
clf.fit(np.concatenate([X_train, X_test]).reshape(-1, 1), 
        np.concatenate([np.zeros(1000), np.ones(500)]))
probs = clf.predict_proba(X_train.reshape(-1, 1))
weights = probs[:, 1] / probs[:, 0]

# 加权回归
model_w = LinearRegression()
model_w.fit(X_train.reshape(-1, 1), y_train, sample_weight=weights)
mse_after = mean_squared_error(y_test, model_w.predict(X_test.reshape(-1, 1)))

print(f"校正前 MSE: {mse_before:.4f}")
print(f"校正后 MSE: {mse_after:.4f}")
print(f"改善: {(mse_before - mse_after)/mse_before*100:.1f}%")

print("\n=== 所有任务完成 ===")


=== 5.2 协变量偏移校正 ===
校正前 MSE: 0.0102
校正后 MSE: 0.0240
改善: -135.9%

=== 所有任务完成 ===
